# Clickbait Detection — Unified Feature Fusion Pipeline

This notebook combines **Lexical (TF-IDF)** + **Syntactic (spaCy POS/dependency numeric features)** + **Semantic (SentenceTransformer embeddings)** into a **single feature matrix** via feature-level fusion.

**Strict scope:** feature extraction + fusion only (no classifier training).

## Dataset (Unified)
We build one canonical dataset from the 2017 training sets:
- `datasets/clickbait17-train-170331` (≈2.4k)
- `datasets/clickbait17-train-170630/clickbait17-validation-170630` (≈19.5k)

Then we shuffle once (fixed seed) and do a single stratified 80/20 split for train/validation so **all three feature blocks see the same rows in the same order**.

**Note on alignment:** the provided semantic notebook reads `clickbait_data.csv` (not present in this repo). Here we keep the *semantic feature logic* (SentenceTransformer encoding) but feed it the unified JSONL dataset to satisfy the “same dataset/order” requirement.

In [2]:
!{sys.executable} -m pip install spacy
!{sys.executable} -m spacy download en_core_web_sm

'{sys.executable}' is not recognized as an internal or external command,
operable program or batch file.
'{sys.executable}' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!{sys.executable} -m pip install sentence-transformers

'{sys.executable}' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import os
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Lexical preprocessing (from Clickbait_Lexical-checkpoint.ipynb)
import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from scipy.sparse import hstack as sparse_hstack
from scipy.sparse import csr_matrix

nltk.download('stopwords', quiet=True)

True

In [5]:
# ----------------------------
# 1) DATA LOADING (UNIFIED)
# ----------------------------

def load_clickbait17_data(base_path: str | os.PathLike) -> pd.DataFrame:
    """
    Load Clickbait17 dataset (2017 format)

    Returns: DataFrame with columns ['headline', 'label']
    Label: (truthMean >= 0.5) -> 1 else 0
    """
    base_path = str(base_path)
    instances_path = os.path.join(base_path, 'instances.jsonl')
    truth_path = os.path.join(base_path, 'truth.jsonl')

    # Stream JSONL line-by-line to keep peak memory low (avoids pandas read_json combining lines).
    truth_mean_by_id: dict[str, float] = {}
    with open(truth_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if 'id' in obj and 'truthMean' in obj:
                truth_mean_by_id[str(obj['id'])] = float(obj['truthMean'])

    rows: list[dict[str, object]] = []
    with open(instances_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            _id = str(obj.get('id', ''))
            if not _id:
                continue
            truth_mean = truth_mean_by_id.get(_id)
            if truth_mean is None:
                continue
            post_text = obj.get('postText')
            headline = post_text[0] if isinstance(post_text, list) and len(post_text) > 0 else str(post_text)
            rows.append({
                'headline': headline,
                'label': 1 if truth_mean >= 0.5 else 0,
            })

    df = pd.DataFrame(rows)
    df = df[['headline', 'label']].dropna().copy()
    return df


class TextPreprocessor:
    """
    Advanced text preprocessing for lexical modeling (from lexical notebook).
    Creates two views:
    - clean_text: for word TF-IDF
    - text_with_punct: for models that can keep punctuation
    """

    def __init__(self, remove_stopwords: bool = False, lowercase: bool = True):
        self.remove_stopwords = remove_stopwords
        self.lowercase = lowercase

        # Custom stopwords (keep important clickbait words)
        base_stopwords = set(stopwords.words('english'))
        important_words = {
            'you', 'your', 'yours', 'what', 'which', 'who', 'whom',
            'this', 'that', 'these', 'those', 'why', 'how'
        }
        self.stop_words = base_stopwords - important_words if remove_stopwords else set()

    def clean_text(self, text: str) -> str:
        if pd.isna(text):
            return ''
        text = str(text)
        if self.lowercase:
            text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if self.remove_stopwords:
            words = text.split()
            words = [w for w in words if w not in self.stop_words]
            text = ' '.join(words)
        return text

    def clean_text_preserve_punctuation(self, text: str) -> str:
        if pd.isna(text):
            return ''
        text = str(text)
        if self.lowercase:
            text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text

In [6]:
# Resolve dataset paths relative to this repo
PROJECT_ROOT = Path.cwd()
DATASETS_DIR = PROJECT_ROOT / 'datasets'

PATH_2017_SMALL = DATASETS_DIR / 'clickbait17-train-170331'
PATH_2017_LARGE = DATASETS_DIR / 'clickbait17-train-170630' / 'clickbait17-validation-170630'

if not PATH_2017_SMALL.exists():
    raise FileNotFoundError(f'Missing: {PATH_2017_SMALL}')
if not PATH_2017_LARGE.exists():
    raise FileNotFoundError(f'Missing: {PATH_2017_LARGE}')

df_2017_small = load_clickbait17_data(PATH_2017_SMALL)
df_2017_large = load_clickbait17_data(PATH_2017_LARGE)

# Combine (matches lexical notebook order: large then small)
df_train_full = pd.concat([df_2017_large, df_2017_small], ignore_index=True)
# Single shuffle so downstream features align identically
df_train_full = df_train_full.sample(frac=1, random_state=42).reset_index(drop=True)

preprocessor = TextPreprocessor(remove_stopwords=False, lowercase=True)
df_train_full['clean_text'] = df_train_full['headline'].apply(preprocessor.clean_text)
df_train_full['text_with_punct'] = df_train_full['headline'].apply(preprocessor.clean_text_preserve_punctuation)

# Drop rows that became empty after cleaning (lexical notebook does this)
df_train_full = df_train_full[df_train_full['clean_text'].str.strip() != ''].reset_index(drop=True)

print('Unified dataset ready')
print('Total samples:', len(df_train_full))
print('Clickbait rate:', df_train_full['label'].mean())
df_train_full.head()

Unified dataset ready
Total samples: 21926
Clickbait rate: 0.24587248016054


,headline,label,clean_text,text_with_punct
0,Expert reveals the 5 best recipes to keep you ...,1,expert reveals the best recipes to keep you sl...,expert reveals the 5 best recipes to keep you ...
1,"A gunman opened fire on the Champs-Élysées, ki...",0,a gunman opened fire on the champs lys es kill...,"a gunman opened fire on the champs-élysées, ki..."
2,What is your university doing to prevent rape?...,0,what is your university doing to prevent rape ...,what is your university doing to prevent rape?...
3,Such dog. Much no :/\n#ThisIsIT,1,such dog much no thisisit,such dog. much no :/ #thisisit
4,Diplomatic assassinations have a long and trag...,1,diplomatic assassinations have a long and trag...,diplomatic assassinations have a long and trag...


In [7]:
# Single consistent split (80/20) for all feature blocks
train_idx, val_idx = train_test_split(
    df_train_full.index.values,
    test_size=0.2,
    random_state=42,
    stratify=df_train_full['label'].values
)

df_train = df_train_full.loc[train_idx].copy()
df_val = df_train_full.loc[val_idx].copy()

# Preserve a deterministic row order inside each split
df_train = df_train.sort_index().reset_index(drop=True)
df_val = df_val.sort_index().reset_index(drop=True)

y_train = df_train['label'].values
y_val = df_val['label'].values

print('Train size:', len(df_train), 'Val size:', len(df_val))
print('Train clickbait rate:', y_train.mean(), 'Val clickbait rate:', y_val.mean())

Train size: 17540 Val size: 4386
Train clickbait rate: 0.2458950969213227 Val clickbait rate: 0.24578203374373006


## 2) Lexical Features (TF-IDF)
Word-level TF-IDF using the same vectorizer settings as the lexical notebook’s final word TF-IDF block.

In [8]:
# Fit TF-IDF on TRAIN only, transform VAL (prevents leakage)
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_lex_train = tfidf_vectorizer.fit_transform(df_train['clean_text'])
X_lex_val = tfidf_vectorizer.transform(df_val['clean_text'])

print('X_lex_train:', X_lex_train.shape, 'type:', type(X_lex_train))
print('X_lex_val:  ', X_lex_val.shape, 'type:', type(X_lex_val))
assert X_lex_train.shape[0] == len(df_train)
assert X_lex_val.shape[0] == len(df_val)

X_lex_train: (17540, 10000) type: <class 'scipy.sparse._csr.csr_matrix'>
X_lex_val:   (4386, 10000) type: <class 'scipy.sparse._csr.csr_matrix'>


## 3) Syntactic Features (spaCy POS/dependency counts + structure)
Numeric features extracted from the syntactic notebook (`syntactic-final.ipynb`).

In [9]:
import spacy

# If needed (first run), install the model in your environment:
#   python -m spacy download en_core_web_sm

nlp = spacy.load('en_core_web_sm', disable=['ner'])

def extract_syntactic_features(texts: pd.Series) -> pd.DataFrame:
    rows = []
    for doc in nlp.pipe(texts.tolist(), batch_size=128):
        pos_tags = [token.pos_ for token in doc]
        # Dependency + head POS (matches syntactic-final notebook)
        dep_patterns = [f'{token.dep_}->{token.head.pos_}' for token in doc]

        pos_counts = {f'pos_{tag}': pos_tags.count(tag) for tag in set(pos_tags)}
        dep_counts = {f'dep_{d}': dep_patterns.count(d) for d in set(dep_patterns)}

        num_tokens = len(doc)
        row = {
            **pos_counts,
            **dep_counts,
            'num_tokens': num_tokens,
            'is_question': int(any(token.text == '?' for token in doc)),
            'num_exclam': doc.text.count('!'),
            'starts_with_pronoun': int(len(doc) > 0 and doc[0].pos_ == 'PRON'),
            'avg_token_len': float(np.mean([len(token.text) for token in doc])) if num_tokens > 0 else 0.0,
        }
        rows.append(row)
    return pd.DataFrame(rows).fillna(0)

X_syn_train_df = extract_syntactic_features(df_train['headline'])
X_syn_val_df = extract_syntactic_features(df_val['headline'])

# Align validation columns to training columns (fill missing with 0)
X_syn_val_df = X_syn_val_df.reindex(columns=X_syn_train_df.columns, fill_value=0)

# Scale (as done in syntactic-final.ipynb)
syn_scaler = StandardScaler()
X_syn_train = syn_scaler.fit_transform(X_syn_train_df)
X_syn_val = syn_scaler.transform(X_syn_val_df)

print('X_syn_train:', X_syn_train.shape, 'dense ndarray')
print('X_syn_val:  ', X_syn_val.shape, 'dense ndarray')
assert X_syn_train.shape[0] == len(df_train)
assert X_syn_val.shape[0] == len(df_val)

X_syn_train: (17540, 429) dense ndarray
X_syn_val:   (4386, 429) dense ndarray


In [10]:
# --- POS sequence TF-IDF (required for X_syntactic_combined) ---
def pos_sequence(texts: pd.Series) -> list[str]:
    return [
        " ".join([token.pos_ for token in doc])
        for doc in nlp.pipe(texts.tolist(), batch_size=128)
    ]

pos_train = pos_sequence(df_train['headline'])
pos_val = pos_sequence(df_val['headline'])

pos_tfidf_vectorizer = TfidfVectorizer(ngram_range=(2, 3), min_df=2)
X_pos_train = pos_tfidf_vectorizer.fit_transform(pos_train)
X_pos_val = pos_tfidf_vectorizer.transform(pos_val)

print('X_pos_train:', X_pos_train.shape, 'type:', type(X_pos_train))
print('X_pos_val:  ', X_pos_val.shape, 'type:', type(X_pos_val))
assert X_pos_train.shape[0] == len(df_train)
assert X_pos_val.shape[0] == len(df_val)

X_pos_train: (17540, 2667) type: <class 'scipy.sparse._csr.csr_matrix'>
X_pos_val:   (4386, 2667) type: <class 'scipy.sparse._csr.csr_matrix'>


## 4) Semantic Features (SentenceTransformer embeddings)
Embeddings extracted with the same model family used in `CLICKBAIT_NLP_PROJECT (3).ipynb` (MiniLM).

In [11]:
from sentence_transformers import SentenceTransformer

# SentenceTransformer embeddings (no fitting; pure encoding)
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

# Use the punctuation-preserving cleaned text for consistency with lexical cleaning
X_sem_train = semantic_model.encode(df_train['text_with_punct'].tolist(), show_progress_bar=True)
X_sem_val = semantic_model.encode(df_val['text_with_punct'].tolist(), show_progress_bar=True)

print('X_sem_train:', X_sem_train.shape, type(X_sem_train))
print('X_sem_val:  ', X_sem_val.shape, type(X_sem_val))
assert X_sem_train.shape[0] == len(df_train)
assert X_sem_val.shape[0] == len(df_val)

Batches: 100%|██████████| 138/138 [00:12<00:00, 10.69it/s]

X_sem_train: (17540, 384) <class 'numpy.ndarray'>
X_sem_val:   (4386, 384) <class 'numpy.ndarray'>


## 5) Feature Combination (Fusion)
Because TF-IDF is sparse, we combine features with `scipy.sparse.hstack`:

$$X_{combined} = [X_{lexical} | X_{syntactic} | X_{semantic}]$$

In [10]:
# Convert dense blocks to sparse before hstack
X_syn_train_sp = csr_matrix(X_syn_train)
X_syn_val_sp = csr_matrix(X_syn_val)
X_sem_train_sp = csr_matrix(X_sem_train)
X_sem_val_sp = csr_matrix(X_sem_val)

# Required syntactic combination: hstack([X_pos_tfidf, X_syn_numeric])
X_syntactic_combined_train = sparse_hstack([X_pos_train, X_syn_train_sp], format='csr')
X_syntactic_combined_val = sparse_hstack([X_pos_val, X_syn_val_sp], format='csr')

# Final feature matrix: hstack([X_lexical, X_syntactic_combined, X_semantic])
X_combined_train = sparse_hstack(
    [X_lex_train, X_syntactic_combined_train, X_sem_train_sp],
    format='csr',
)
X_combined_val = sparse_hstack(
    [X_lex_val, X_syntactic_combined_val, X_sem_val_sp],
    format='csr',
)

print('X_syntactic_combined_train:', X_syntactic_combined_train.shape, 'type:', type(X_syntactic_combined_train))
print('X_syntactic_combined_val:  ', X_syntactic_combined_val.shape, 'type:', type(X_syntactic_combined_val))
print('X_combined_train:', X_combined_train.shape, 'type:', type(X_combined_train))
print('X_combined_val:  ', X_combined_val.shape, 'type:', type(X_combined_val))

assert X_syntactic_combined_train.shape[0] == len(y_train)
assert X_syntactic_combined_val.shape[0] == len(y_val)
assert X_combined_train.shape[0] == len(y_train)
assert X_combined_val.shape[0] == len(y_val)
assert X_combined_train.shape[1] == X_combined_val.shape[1]

# Primary outputs requested by the spec
X_combined = X_combined_train
y = y_train

print('Ready for modeling: X_combined, y (train split)')

X_syntactic_combined_train: (17540, 3096) type: <class 'scipy.sparse._csr.csr_matrix'>
X_syntactic_combined_val:   (4386, 3096) type: <class 'scipy.sparse._csr.csr_matrix'>
X_combined_train: (17540, 13480) type: <class 'scipy.sparse._csr.csr_matrix'>
X_combined_val:   (4386, 13480) type: <class 'scipy.sparse._csr.csr_matrix'>
Ready for modeling: X_combined, y (train split)


In [43]:

# D - lexical + syntactic_combined
# (uses POS TF-IDF + scaled syntactic numeric inside X_syntactic_combined_*)
X_lex_syn_train = sparse_hstack([X_lex_train, X_syntactic_combined_train], format='csr')
X_lex_syn_val = sparse_hstack([X_lex_val, X_syntactic_combined_val], format='csr')

# E - lexical + semantic
X_lex_sem_train = sparse_hstack([X_lex_train, X_sem_train_sp], format='csr')
X_lex_sem_val = sparse_hstack([X_lex_val, X_sem_val_sp], format='csr')

print('X_lex_syn_train:', X_lex_syn_train.shape)
print('X_lex_syn_val:  ', X_lex_syn_val.shape)
print('X_lex_sem_train:', X_lex_sem_train.shape)
print('X_lex_sem_val:  ', X_lex_sem_val.shape)

assert X_lex_syn_train.shape[0] == len(y_train)
assert X_lex_syn_val.shape[0] == len(y_val)
assert X_lex_sem_train.shape[0] == len(y_train)
assert X_lex_sem_val.shape[0] == len(y_val)

X_lex_syn_train: (17540, 13096)
X_lex_syn_val:   (4386, 13096)
X_lex_sem_train: (17540, 10384)
X_lex_sem_val:   (4386, 10384)


## 6) Baseline Models (Training + Validation Eval)
Evaluate baseline classifiers on feature combinations using **existing matrices only**:
- **D:** Lexical + Syntactic
- **E:** Lexical + Semantic
- **G:** Lexical + Syntactic + Semantic (full combined)

In [45]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from scipy import sparse

# ---------------------------------------
# STEP 1: Identify existing variables
# ---------------------------------------
required_vars = [
    'X_lex_train', 'X_lex_val',
    'X_syntactic_combined_train', 'X_syntactic_combined_val',
    'X_sem_train_sp', 'X_sem_val_sp',
    'X_combined_train', 'X_combined_val',
    'X_lex_syn_train', 'X_lex_syn_val',
    'X_lex_sem_train', 'X_lex_sem_val',
    'y_train', 'y_val',
 ]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(f"Missing required notebook variables: {missing}")

print('=== Existing Variables (shapes) ===')
print('X_lex_train:', X_lex_train.shape)
print('X_lex_val:  ', X_lex_val.shape)
print('X_syntactic_combined_train:', X_syntactic_combined_train.shape)
print('X_syntactic_combined_val:  ', X_syntactic_combined_val.shape)
print('X_sem_train_sp:', X_sem_train_sp.shape)
print('X_sem_val_sp:  ', X_sem_val_sp.shape)
print('X_combined_train:', X_combined_train.shape)
print('X_combined_val:  ', X_combined_val.shape)
print('y_train:', y_train.shape)
print('y_val:  ', y_val.shape)

# ---------------------------------------
# STEP 2: Feature combinations (D, E, G)
# ---------------------------------------
# D: Lexical + Syntactic
X_D_train, X_D_val = X_lex_syn_train, X_lex_syn_val

# E: Lexical + Semantic
X_E_train, X_E_val = X_lex_sem_train, X_lex_sem_val

# G: All features (already created full combined features)
X_G_train, X_G_val = X_combined_train, X_combined_val

feature_sets = [
    ('Lex + Syn', X_D_train, X_D_val),
    ('Lex + Sem', X_E_train, X_E_val),
    ('All Features', X_G_train, X_G_val),
 ]

# ---------------------------------------
# STEP 3-4: Models (no tuning) + Evaluation
# ---------------------------------------
def _is_sparse(X):
    return sparse.issparse(X)

def _min_value(X):
    # Works for both dense and sparse; includes implicit zeros for sparse matrices.
    return float(X.min()) if _is_sparse(X) else float(X.min())

def eval_one(feature_name: str, model, X_tr, X_va) -> None:
    # Strict rule: do not convert sparse -> dense.
    if isinstance(model, MLPClassifier) and _is_sparse(X_tr):
        print(f"\n[{feature_name}]\nSKIPPED: MLPClassifier does not support sparse input (dense conversion disallowed).")
        return

    # MultinomialNB requires non-negative features.
    if isinstance(model, MultinomialNB):
        if _min_value(X_tr) < 0 or _min_value(X_va) < 0:
            print(f"\n[{feature_name}]\nSKIPPED: MultinomialNB requires non-negative features, but this feature set contains negative values.")
            return

    try:
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_va)
    except Exception as e:
        print(f"\n[{feature_name}]\nERROR during fit/predict: {type(e).__name__}: {e}")
        return

    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, zero_division=0)
    rec = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    print(f"\n[{feature_name}]")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(classification_report(y_val, y_pred, digits=4, zero_division=0))

models = [
    ('Logistic Regression (default)', LogisticRegression(max_iter=1000)),
    ('Logistic Regression (balanced)', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ('Linear SVM (default)', LinearSVC()),
    ('Linear SVM (balanced)', LinearSVC(class_weight='balanced')),
    ('SGDClassifier hinge (default)', SGDClassifier(loss='hinge')),
    ('SGDClassifier hinge (balanced)', SGDClassifier(loss='hinge', class_weight='balanced')),
    ('MLPClassifier', MLPClassifier(hidden_layer_sizes=(100,), max_iter=20)),
    ('MultinomialNB', MultinomialNB()),
 ]

for model_name, model in models:
    print(f"\n=== Model: {model_name} ===")
    for feat_name, X_tr, X_va in feature_sets:
        eval_one(feat_name, model, X_tr, X_va)

=== Existing Variables (shapes) ===
X_lex_train: (17540, 10000)
X_lex_val:   (4386, 10000)
X_syntactic_combined_train: (17540, 3096)
X_syntactic_combined_val:   (4386, 3096)
X_sem_train_sp: (17540, 384)
X_sem_val_sp:   (4386, 384)
X_combined_train: (17540, 13480)
X_combined_val:   (4386, 13480)
y_train: (17540,)
y_val:   (4386,)

=== Model: Logistic Regression (default) ===

[Lex + Syn]
Accuracy:  0.8468
Precision: 0.7428
Recall:    0.5761
F1-score:  0.6489
              precision    recall  f1-score   support

           0     0.8713    0.9350    0.9020      3308
           1     0.7428    0.5761    0.6489      1078

    accuracy                         0.8468      4386
   macro avg     0.8070    0.7555    0.7755      4386
weighted avg     0.8397    0.8468    0.8398      4386


[Lex + Sem]
Accuracy:  0.8463
Precision: 0.7596
Recall:    0.5482
F1-score:  0.6369
              precision    recall  f1-score   support

           0     0.8650    0.9435    0.9025      3308
           1     

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score
import joblib

# ---------------------------------------
# FINAL MODEL COMPARISON (TUNING)
# Uses ONLY: X_combined_train, X_combined_val, y_train, y_val
# ---------------------------------------
required_vars = ['X_combined_train', 'X_combined_val', 'y_train', 'y_val']
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(f"Missing required notebook variables: {missing}")

# STEP 1: Logistic Regression tuning
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10],
    'class_weight': [None, 'balanced'],
    'solver': ['liblinear'],
}

lr = LogisticRegression(max_iter=1000)
grid_lr = GridSearchCV(
    lr,
    param_grid_lr,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    pre_dispatch=1,
 )

# Use threading backend to avoid high-memory multiprocessing copies on large sparse matrices
with joblib.parallel_backend('threading'):
    grid_lr.fit(X_combined_train, y_train)
best_lr = grid_lr.best_estimator_

# STEP 2: Linear SVM tuning
param_grid_svm = {
    'C': [0.01, 0.1, 1, 10],
    'loss': ['hinge', 'squared_hinge'],
    'class_weight': [None, 'balanced'],
}

# NOTE: dual=True is required for loss='hinge' (keeps grid valid without changing features).
svm = LinearSVC(dual=True)
grid_svm = GridSearchCV(
    svm,
    param_grid_svm,
    cv=3,
    scoring='f1',
    pre_dispatch=1,
 )
grid_svm.fit(X_combined_train, y_train)
best_svm = grid_svm.best_estimator_

# STEP 3: Evaluation on validation split
y_pred_lr = best_lr.predict(X_combined_val)
y_pred_svm = best_svm.predict(X_combined_val)

f1_lr = f1_score(y_val, y_pred_lr)
f1_svm = f1_score(y_val, y_pred_svm)

# STEP 4: Output
print('=== FINAL MODEL COMPARISON (TUNED) ===')
print('\nBest LR parameters:', grid_lr.best_params_)
print('Validation F1 (LR):', f"{f1_lr:.4f}")
print('\nBest SVM parameters:', grid_svm.best_params_)
print('Validation F1 (SVM):', f"{f1_svm:.4f}")

print('\n=== Best Logistic Regression: classification_report ===')
print(classification_report(y_val, y_pred_lr, digits=4, zero_division=0))

print('\n=== Best Linear SVM (LinearSVC): classification_report ===')
print(classification_report(y_val, y_pred_svm, digits=4, zero_division=0))

=== FINAL MODEL COMPARISON (TUNED) ===

Best LR parameters: {'C': 1, 'class_weight': 'balanced', 'solver': 'liblinear'}
Validation F1 (LR): 0.6836

Best SVM parameters: {'C': 0.1, 'class_weight': 'balanced', 'loss': 'squared_hinge'}
Validation F1 (SVM): 0.6894

=== Best Logistic Regression: classification_report ===
              precision    recall  f1-score   support

           0     0.9129    0.8555    0.8833      3308
           1     0.6283    0.7495    0.6836      1078

    accuracy                         0.8295      4386
   macro avg     0.7706    0.8025    0.7834      4386
weighted avg     0.8430    0.8295    0.8342      4386


=== Best Linear SVM (LinearSVC): classification_report ===
              precision    recall  f1-score   support

           0     0.9161    0.8552    0.8846      3308
           1     0.6310    0.7597    0.6894      1078

    accuracy                         0.8317      4386
   macro avg     0.7735    0.8075    0.7870      4386
weighted avg     0.8460

In [18]:
# ---------------------------------------
# 2017 TEST SET EVALUATION (clickbait17-test-170720)
# Uses existing preprocessing/feature extractors (transform/encode only)
# Assumes previous cells already ran and created:
#   tfidf_vectorizer, pos_tfidf_vectorizer, syn_scaler, X_syn_train_df.columns, semantic_model, best_lr, best_svm
# ---------------------------------------
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

PATH_2017_TEST = DATASETS_DIR / 'clickbait17-test-170720'
if not PATH_2017_TEST.exists():
    raise FileNotFoundError(f'Missing: {PATH_2017_TEST}')

# Load test set (has labels in truth.jsonl)
df_test_2017 = load_clickbait17_data(PATH_2017_TEST).copy()
df_test_2017['clean_text'] = df_test_2017['headline'].apply(preprocessor.clean_text)
df_test_2017['text_with_punct'] = df_test_2017['headline'].apply(preprocessor.clean_text_preserve_punctuation)

before = len(df_test_2017)
df_test_2017 = df_test_2017[df_test_2017['clean_text'].str.strip() != ''].reset_index(drop=True)
after = len(df_test_2017)
if after != before:
    print(f'Dropped empty after cleaning: {before - after}')

y_test_2017 = df_test_2017['label'].values
print('2017 TEST size:', len(df_test_2017), 'clickbait rate:', float(y_test_2017.mean()))

# --- Lexical (TF-IDF) ---
X_lex_test = tfidf_vectorizer.transform(df_test_2017['clean_text'])

# --- POS TF-IDF ---
pos_test = pos_sequence(df_test_2017['headline'])
X_pos_test = pos_tfidf_vectorizer.transform(pos_test)

# --- Numeric syntactic (scaled) ---
X_syn_test_df = extract_syntactic_features(df_test_2017['headline'])
X_syn_test_df = X_syn_test_df.reindex(columns=X_syn_train_df.columns, fill_value=0)
X_syn_test = syn_scaler.transform(X_syn_test_df)
X_syn_test_sp = csr_matrix(X_syn_test)

# Syntactic combined: POS TF-IDF + numeric syntactic
X_syntactic_combined_test = sparse_hstack([X_pos_test, X_syn_test_sp], format='csr')

# --- Semantic (SentenceTransformer) ---
X_sem_test = semantic_model.encode(df_test_2017['text_with_punct'].tolist(), show_progress_bar=True)
X_sem_test_sp = csr_matrix(X_sem_test)

# --- Final combined (lex + syntactic_combined + semantic) ---
X_combined_test = sparse_hstack([X_lex_test, X_syntactic_combined_test, X_sem_test_sp], format='csr')
print('X_combined_test:', X_combined_test.shape, type(X_combined_test))

def _report(name: str, model, X_te, y_te):
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)
    print(f"\n=== 2017 TEST: {name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(classification_report(y_te, y_pred, digits=4, zero_division=0))

_report('Logistic Regression (tuned best_lr)', best_lr, X_combined_test, y_test_2017)
_report('Linear SVM (tuned best_svm)', best_svm, X_combined_test, y_test_2017)

Dropped empty after cleaning: 89
2017 TEST size: 18890 clickbait rate: 0.22906299629433563


Batches: 100%|██████████| 591/591 [01:20<00:00,  7.34it/s]


X_combined_test: (18890, 13480) <class 'scipy.sparse._csr.csr_matrix'>

=== 2017 TEST: Logistic Regression (tuned best_lr) ===
Accuracy:  0.8336
Precision: 0.6015
Recall:    0.8103
F1-score:  0.6904
              precision    recall  f1-score   support

           0     0.9371    0.8405    0.8862     14563
           1     0.6015    0.8103    0.6904      4327

    accuracy                         0.8336     18890
   macro avg     0.7693    0.8254    0.7883     18890
weighted avg     0.8603    0.8336    0.8413     18890


=== 2017 TEST: Linear SVM (tuned best_svm) ===
Accuracy:  0.8349
Precision: 0.6037
Recall:    0.8123
F1-score:  0.6927
              precision    recall  f1-score   support

           0     0.9379    0.8416    0.8871     14563
           1     0.6037    0.8123    0.6927      4327

    accuracy                         0.8349     18890
   macro avg     0.7708    0.8270    0.7899     18890
weighted avg     0.8613    0.8349    0.8426     18890



In [20]:
# ---------------------------------------
# 2016 WEBIS CROSS-DATASET EVALUATION (corpus-webis-clickbait-16)
# DO NOT fit/refit anything: ONLY load, preprocess, transform, stack, evaluate.
# Parsing logic mirrors syntactic-final.ipynb: truth/majority.csv + problems/*/*.json
# ---------------------------------------
import os
import json

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

PATH_2016 = DATASETS_DIR / 'corpus-webis-clickbait-16' / 'webis-clickbait-16'
if not PATH_2016.exists():
    raise FileNotFoundError(f'Missing: {PATH_2016}')

problems_path = PATH_2016 / 'problems'
truth_file = PATH_2016 / 'truth' / 'majority.csv'
if not problems_path.exists():
    raise FileNotFoundError(f'Missing: {problems_path}')
if not truth_file.exists():
    raise FileNotFoundError(f'Missing: {truth_file}')

# --- STEP 0: Load Webis 2016 headlines + labels (binary) ---
truth_df = pd.read_csv(truth_file, header=None, dtype={0: str})
truth_df.columns = ['id', 'label_str']
truth_map = dict(zip(truth_df['id'], truth_df['label_str']))

data_2016 = []
for folder in sorted(os.listdir(problems_path)):
    folder_path = problems_path / folder
    if not folder_path.is_dir():
        continue
    label_str = truth_map.get(folder)
    if label_str is None:
        continue
    label = 1 if label_str == 'clickbait' else 0

    json_files = sorted([p for p in folder_path.iterdir() if p.suffix == '.json'])
    if not json_files:
        continue

    json_path = json_files[0]
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            problem = json.load(f)
    except Exception:
        continue

    headline = None
    if 'postText' in problem:
        val = problem['postText']
        headline = val[0] if isinstance(val, list) and len(val) > 0 else val
    elif 'text' in problem:
        headline = problem['text']
    elif 'title' in problem:
        headline = problem['title']

    if headline and isinstance(headline, str):
        data_2016.append({'headline': headline, 'label': label})

df_2016 = pd.DataFrame(data_2016, columns=['headline', 'label']).dropna().reset_index(drop=True)
print('2016 WEBIS size:', len(df_2016), 'clickbait rate:', float(df_2016['label'].mean()))

# --- STEP 1: Preprocessing (MUST match training) ---
df_2016['clean_text'] = df_2016['headline'].apply(preprocessor.clean_text)
df_2016['text_with_punct'] = df_2016['headline'].apply(preprocessor.clean_text_preserve_punctuation)

before = len(df_2016)
df_2016 = df_2016[df_2016['clean_text'].str.strip() != ''].reset_index(drop=True)
after = len(df_2016)
if after != before:
    print(f'Dropped empty after cleaning: {before - after}')

y_2016 = df_2016['label'].values

# --- STEP 2: Feature transforms (NO fitting) ---
X_lex_2016 = tfidf_vectorizer.transform(df_2016['clean_text'])

# Compute POS sequences + syntactic numeric features in ONE spaCy pass (same logic, smaller batches to avoid OOM).
pos_2016 = []
syn_rows_2016 = []
for doc in nlp.pipe(df_2016['headline'].tolist(), batch_size=16):
    pos_tags = [token.pos_ for token in doc]
    pos_2016.append(' '.join(pos_tags))

    dep_patterns = [f'{token.dep_}->{token.head.pos_}' for token in doc]
    pos_counts = {f'pos_{tag}': pos_tags.count(tag) for tag in set(pos_tags)}
    dep_counts = {f'dep_{d}': dep_patterns.count(d) for d in set(dep_patterns)}

    num_tokens = len(doc)
    row = {
        **pos_counts,
        **dep_counts,
        'num_tokens': num_tokens,
        'is_question': int(any(token.text == '?' for token in doc)),
        'num_exclam': doc.text.count('!'),
        'starts_with_pronoun': int(len(doc) > 0 and doc[0].pos_ == 'PRON'),
        'avg_token_len': float(np.mean([len(token.text) for token in doc])) if num_tokens > 0 else 0.0,
    }
    syn_rows_2016.append(row)

X_pos_2016 = pos_tfidf_vectorizer.transform(pos_2016)

X_syn_2016_df = pd.DataFrame(syn_rows_2016).fillna(0)
X_syn_2016_df = X_syn_2016_df.reindex(columns=X_syn_train_df.columns, fill_value=0)
X_syn_2016 = syn_scaler.transform(X_syn_2016_df)
X_syn_2016_sp = csr_matrix(X_syn_2016)

X_syn_combined_2016 = sparse_hstack([X_pos_2016, X_syn_2016_sp], format='csr')

X_sem_2016 = semantic_model.encode(df_2016['text_with_punct'].tolist(), show_progress_bar=True)
X_sem_2016_sp = csr_matrix(X_sem_2016)

# --- STEP 3: Final combined matrix (lex + syntactic_combined + semantic) ---
X_combined_2016 = sparse_hstack([X_lex_2016, X_syn_combined_2016, X_sem_2016_sp], format='csr')
print('X_combined_2016:', X_combined_2016.shape, type(X_combined_2016))

# --- STEP 4: Alignment sanity checks ---
assert X_combined_2016.shape[0] == len(y_2016)
assert X_combined_2016.shape[1] == X_combined_train.shape[1], (
    f'Feature dimension mismatch: 2016={X_combined_2016.shape[1]} vs train={X_combined_train.shape[1]}'
 )

# --- STEP 5: Evaluate using already-trained tuned models ---
def _report(name: str, model, X_te, y_te):
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)
    print(f"\n=== 2016 WEBIS: {name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(classification_report(y_te, y_pred, digits=4, zero_division=0))

_report('Logistic Regression (tuned best_lr)', best_lr, X_combined_2016, y_2016)
_report('Linear SVM (tuned best_svm)', best_svm, X_combined_2016, y_2016)

2016 WEBIS size: 2992 clickbait rate: 0.25635026737967914


Batches: 100%|██████████| 94/94 [00:16<00:00,  5.62it/s]


X_combined_2016: (2992, 13480) <class 'scipy.sparse._csr.csr_matrix'>

=== 2016 WEBIS: Logistic Regression (tuned best_lr) ===
Accuracy:  0.7998
Precision: 0.6024
Recall:    0.6441
F1-score:  0.6226
              precision    recall  f1-score   support

           0     0.8743    0.8535    0.8638      2225
           1     0.6024    0.6441    0.6226       767

    accuracy                         0.7998      2992
   macro avg     0.7384    0.7488    0.7432      2992
weighted avg     0.8046    0.7998    0.8019      2992


=== 2016 WEBIS: Linear SVM (tuned best_svm) ===
Accuracy:  0.8045
Precision: 0.6099
Recall:    0.6584
F1-score:  0.6332
              precision    recall  f1-score   support

           0     0.8789    0.8548    0.8667      2225
           1     0.6099    0.6584    0.6332       767

    accuracy                         0.8045      2992
   macro avg     0.7444    0.7566    0.7500      2992
weighted avg     0.8100    0.8045    0.8069      2992

